# Lifting and Pre-processing
While Operator Inference (OpInf) is effective at learning reduced-order models (ROMs), it assumes the governing equations of your system have a distinct polynomial structure. Real-world physical systems are rarely willing to cooperate with this requirement.
To build high-fidelity ROMs using the {mod}`opinf` package, it is important to find a variable transformation to lift the equations to the proper format. This technique is referred to in the literature as **lifting**.

Additionally, systems often have state variables which operate on vastly different scales. Since OpInf relies on least-squares regression to approximate the reduced-order model (ROM), failing to correctly **scale** the input data leads to a weak model.

The purpose of this tutorial is to become acquainted with the tools and techniques required for lifting and scaling to increase the fidelity of our ROM. We also show the dire consequences of trying to learn a ROM from raw data.

## Problem Statement

:::{admonition} Governing Equations
:class: note
**Compressible Euler Flow of an Ideal Gas (1D)**

Let $\Omega = [0,L]\subset \mathbb{R}$ be the spatial domain indicated by the variable $x$, and let $[t_0,t_\text{final}]\subset\mathbb{R}$ be the time domain with variable $t$. We consider the one-dimensional Euler equations for the compressible flow of an ideal gas with periodic boundary conditions.
The state is given by

$$
\begin{aligned}
    \vec{q}_\text{c}(x, t) = \left[\begin{array}{c}
        \rho \\ \rho v \\ \rho e
    \end{array}\right],
\end{aligned}
$$

where $\rho = \rho(x,t)$ is the density $[\frac{\text{kg}}{\text{m}^3}]$, $v = v(x,t)$ is the fluid velocity $[\frac{\text{m}}{\text{s}}]$, and $e = e(x, t)$ is the internal energy per unit mass $[\frac{\text{m}^2}{\text{s}^2}]$.
The state evolves in time according to the following conservative system of partial differential equations (PDEs):

$$
\begin{aligned}
    \frac{\partial\vec{q}_\text{c}}{\partial t}
    = \frac{\partial}{\partial t} \left[\begin{array}{c}
        \rho \\ \rho v \\ \rho e
    \end{array}\right]
    &= -\frac{\partial}{\partial x}\left[\begin{array}{c}
        \rho v \\ \rho v^2 + p \\ (\rho e + p) v
    \end{array}\right]
    & x &\in\Omega,\quad t\in[t_0,t_\text{final}], 
    \\
    \vec{q}_\text{c}(0,t) &= \vec{q}_\text{c}(L,t)
    & t &\in[t_0,t_\text{final}],
    \\
    \vec{q}_\text{c}(x,t_0) &= \vec{q}_{\text{c},0}(x)
    & x &\in \Omega,
\end{aligned}
$$ (eq_lifting_fom)

where $p = p(x,t)$ is the pressure $[\text{Pa}] = [\frac{\text{kg}}{\text{m}\cdot\text{s}^2}]$ and $\vec{q}_{\text{c},0}(x)$ is a given initial condition.
The state variables are related via the ideal gas law

$$
\begin{aligned}
    \rho e = \frac{p}{\gamma - 1} + \frac{1}{2}\rho v^{2},
\end{aligned}
$$ (eq_ideal_gas_law)

where $\gamma = 1.4$ is the nondimensional heat capacity ratio.

**Important**: Note that the dynamics are nonpolynomially nonlinear with respect to $\rho,$ $\rho v,$ and $\rho e$.
:::

:::{admonition} Objective
:class: note
Our objective is to construct a reduced-order model (ROM) which can be solved rapidly to produce approximate solutions to the partial differential equation {eq}`eq_lifting_fom` for various choices of the initial condition $\vec{q}_{\text{c},0}$.
We will only use data observed over a limited time interval $[t_0, t_\text{obs}]$ with $t_\text{obs} < t_\text{final}$ to train the ROM, but the ROM will be used to predict the solution for the entire time domain $[t_0, t_\text{final}]$.
Hence, the ROM will be **predictive in time** and **predictive in the initial conditions**.
:::

We make use of the following standard scientific Python libraries and the {mod}`opinf` package.

In [ ]:
import time
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

import opinf

:::{admonition} In Practice
:class: tip

For pedagogical reasons, this notebook uses the modules of the {mod}`opinf` package one at a time to lift, scale, and compress training data, then assemble and solve an inference problem. In practice, the [`opinf.ROM`](opinf.roms.ROM) class can be used to accomplish the entire sequence in a streamlined way without requiring the user to manage the intermediate steps, which is demonstrated toward the end of this notebook.

:::

## Acquire Training Data

### Full-order Model

A full-order model (FOM) for the PDE is a system of $n = 3n_x$ ODEs that defines evolution equations for the spatially discretized state $\mathbf{q}_{c}(t)$.
We implement a FOM by approximating the spatial derivative $\frac{\partial}{\partial x}$ with a first-order backward finite difference approximation,

$$
\begin{aligned}
    \frac{\partial}{\partial x}y(x,t)
    \approx \frac{y(x, t) - y(x - \delta x, t)}{\delta x},
\end{aligned}
$$

for $y = \rho v,$ $\rho v^2 + p,$ and $(\rho e + p) v.$
The resulting system is integrated in time using an adaptive fourth/fifth order Runge–Kutta method.

For this experiment, training states were generated for several different initial conditions, determined by fixing the pressure at $10^5~[\text{Pa}]$ and constructing splines for the density and the velocity.

:::{dropdown} Discretization details

To generate data, we first define an equidistant grid $\{x_i\}_{i=0}^{n_x}$ over the one-dimensional spatial domain $\Omega$:

$$
\begin{aligned}
    0 &= x_0 < x_1 < \cdots < x_{n_x-1} < x_{n_x} = L,
    &
    \delta x &= \frac{L}{n_x} = x_{i+1} - x_{i},\quad i=0,\ldots,n_x-1.
\end{aligned}
$$

The spatially discretized state vector collects the values of the three state variables at each point in the spatial domain. Because periodic boundary conditions prescribe $q_\text{c}(x_0,t) = q_\text{c}(x_{n_x},t)$, values at the endpoint $x = x_{n_x} = L$ are not included.

$$
\begin{aligned}
    \mathbf{q}_\text{c}(t)
    = \left[\begin{array}{c}
        \\
        \boldsymbol{\rho}(t)
        \\ \\ \hline \\
        \boldsymbol{\rho}\mathbf{v}(t)
        \\ \\ \hline \\
        \boldsymbol{\rho}\mathbf{e}(t)
        \\ \phantom{.}
    \end{array}\right]
    = \left[\begin{array}{c}
        \\
        \rho(x_{0}, t) \\ \vdots \\ \rho(x_{n_{x}-1}, t)
        \\ \\ \hline \\
        (\rho v)(x_{0}, t) \\ \vdots \\ (\rho v)(x_{n_{x}-1}, t)
        \\ \\ \hline \\
        (\rho e)(x_{0}, t) \\ \vdots \\ (\rho e)(x_{n_{x}-1}, t)
        \\ \phantom{.}
    \end{array}\right]
    \in\mathbb{R}^{3n_x}.
\end{aligned}
$$

Our overall goal is to approximate $\mathbf{q}_\text{c}(t)$ over a time grid, which we take to be an equidistant grid of $n_t$ instances starting at $t = 0$:

$$
\begin{aligned}
    0 &= t_0 < t_1 < \cdots < t_{n_t - 1} = t_\text{final},
    &
    \delta t &= \frac{t_\text{final}}{n_t - 1} = t_{j+1} - t_{j}, \quad j=0,\ldots,n_t-2.
\end{aligned}
$$

The next code block loads the experiment data. If you want to follow along locally, you will need to click the button to download `lifting_data.h5` and then place it in the same directory as this tutorial. 
Note carefully that we **only load data** – the outputs of the FOM – as opposed to loading the FOM itself.

:::{button-link} https://github.com/operator-inference/opinf/raw/refs/heads/data/lifting_data.h5
:color: success
:outline:
To download the data, click here.
:::

In [ ]:
data = {} # Load data from h5 file
with opinf.utils.hdf5_loadhandle("lifting_data.h5") as hf:
    for key in ("gamma", "x", "t_train", "training_snapshots", "t_all", "test_init", "test_solution"):
        data[key] = hf[key][:]

gamma = data["gamma"]  # Heat capacity ratio

x = data["x"]  # Spatial domain
dx = x[1] - x[0]  # Spatial step size

t_all = data["t_all"]  # Full time domain of interest
dt = t_all[1] - t_all[0]  # Time step size

t_train = data["t_train"]  # Shorter time domain for training data
Q_fom = data["training_snapshots"]  # Training states, defined over `t_train`.

print(
    f"\nSpatial domain:\t\t[{x[0]}, {x[-1]}] with {x.size} spatial points",
    f"Spatial step size:\t{dx=:.3f}",
    f"\nFull time domain:\t[{t_all[0]}, {t_all[-1]}] "
    f"with {t_all.size} time instances",
    f"Training time domain:\t[{t_train[0]}, {t_train[-1]}] "
    f"with {t_train.size} time instances",
    f"Time step size:\t\t{dt=:.5f}",
    f"\n{Q_fom.shape[0]} training trajectories "
    f"of {Q_fom.shape[-1]} snapshots each",
    sep="\n",
)

| Name                               | Symbol             | Code Variable |
| :--------------------------------- | :----------------- | :-------|
| State Snapshots | $\mathbf{Q}$ | `Q_fom` |
| Spatial Domain | $\Omega$ | `x` |
| Spatial Step Size | $\Delta x$ | `dx`|
| Time Domain    | $[t_0,t_{\text{final}}]$ | `t_all` |
| Training Subset of Time Domain | $[t_0,t_{\text{obs}}]$ | `t_train`|
| Temporal Step Size | $\Delta t$ |`dt`|

### Sanity Check: Visualize Training Data

Before learning a model from data, it is always a good idea to visualize the data and check that it makes sense physically.

In [ ]:
plt.rc("axes", titlesize="x-large", labelsize="x-large")
plt.rc("axes.spines", right=False, top=False)
plt.rc("figure", figsize=(12, 6), dpi=300)
plt.rc("legend", edgecolor="none", frameon=False, fontsize="xx-large")
plt.rc("font", family="serif")
plt.rc("text", usetex=True)

def _euler_ylabels(fig, axes, lifted: bool = False):
    """Add y-labels to Euler state plots."""
    axes[0].set_ylabel(r"$v$" if lifted else r"$\rho$")
    axes[1].set_ylabel(r"$p$" if lifted else r"$\rho v$")
    axes[2].set_ylabel(r"$1/\rho$" if lifted else r"$\rho e$")
    fig.align_ylabels(axes)

def plot_initial_conditions(x, trajectories, ncol: int = 3):
    """Plot three sets of initial conditions over the spatial domain."""
    q0s = [Q[:, 0] for Q in trajectories]
    colors = ['C0', 'C1', 'C2']
    fig, axes = plt.subplots(
        3,
        ncol,
        figsize=(4 * ncol, 6),
        sharex=True,
        sharey="row",
        squeeze=False,
    )
    for i, idx in enumerate(
        np.sort(np.random.choice(len(q0s), size=ncol, replace=False))
    ):
        q0 = q0s[idx]
        for ax, var, c in zip(axes[:, i], np.split(q0, 3, axis=0), colors):
            ax.plot(x, var, color=c)
            ax.set_xlim(x[0], x[-1])
        axes[0, i].set_title(f"Initial condition {idx}")
        axes[-1, i].set_xlabel(r"$x\in[0,L]$")
    _euler_ylabels(fig, axes[:, 0])
    fig.tight_layout()

    return fig, axes

plot_initial_conditions(x, Q_fom)
plt.show()

In [ ]:
import matplotlib.colors
def plot_traces(x, t, states, nlocs: int = 20, lifted: bool = False):
    """Plot traces in time at ``nlocs`` locations."""
    # Choose spatial locations at which to plot each state.
    xlocs = np.linspace(0, states.shape[0] // 3, nlocs + 1, dtype=int)[:-1]
    xlocs += xlocs[1] // 2
    # colors = plt.cm.twilight(np.linspace(0, 1, nlocs + 1)[:-1])
    colors = plt.cm.viridis_r(np.linspace(0, 0.8,nlocs + 1)[:-1])

    # Plot the states in time.
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(12, 6))
    for i, c in zip(xlocs, colors):
        for ax, var in zip(axes, np.split(states, 3, axis=0)):
            ax.plot(t, var[i], color=c, lw=1)

    # Format axes.
    axes[2].set_xlabel(r"time $t$")
    axes[2].set_xlim(t[0], t[-1])
    _euler_ylabels(fig, axes, lifted)

    # Colorbar.
    lsc = plt.cm.viridis_r(np.linspace(0, 1, 400))
    scale = matplotlib.colors.Normalize(vmin=0, vmax=1)
    lscmap = matplotlib.colors.LinearSegmentedColormap.from_list(
        "euler", lsc, N=nlocs
    )
    mappable = plt.cm.ScalarMappable(norm=scale, cmap=lscmap)
    cbar = fig.colorbar(mappable, ax=axes, pad=0.015)
    cbar.set_ticks(x[xlocs] / (x[-1] - x[0]))
    cbar.set_ticklabels([f"{xx:.2f}" for xx in x[xlocs]])
    cbar.set_label(r"spatial coordinate $x$")

    return fig, axes

fig, axes = plot_traces(x, t_train, Q_fom[0])
axes[0].set_title("Training trajectory with initial condition 0")
plt.show()

## Pre-Process Training Data

The data for this problem requires some pre-processing before applying a dimensionality reduction technique. However, not every problem requires these steps, and the type of preprocessing that is most appropriate for another problem may be different than what is presented here. The preprocessing steps that we cover in this case are **lifting** and **scaling**.

### Lift to a Polynomial Form

Recall that the original Euler equations are currently in the form:

$$
\begin{array}{ccc}
\frac{\partial}{\partial t}[\rho] = -\frac{\partial}{\partial x}[\rho u]
&
\frac{\partial}{\partial t}[\rho u] = -\frac{\partial}{\partial x}[\rho u^2 + p]
&
\frac{\partial}{\partial t}[\rho e] = -\frac{\partial}{\partial x}[(\rho e + p)u]
\end{array}
$$

Since these equations are not quadratic, we are unable to use the operator inference framework on them. However, we can perform a change of variables so that they are quadratic. By changing our equation to be in terms of $\vec{q} = (v, p, \zeta)$, where $\zeta = 1/\rho$ is the specific volume $[\text{m}^3/\text{kg}]$. We now have the equations:

$$
\frac{\partial v}{\partial t} = -v \frac{\partial v}{\partial x} - \xi \frac{\partial p}{\partial x}, \quad
\frac{\partial p}{\partial t} = -\gamma p \frac{\partial v}{\partial x} - v \frac{\partial p}{\partial x}, \quad
\frac{\partial \xi}{\partial t} = -v \frac{\partial \xi}{\partial x} + \xi \frac{\partial v}{\partial x}.
$$ (eq_lifted_pdes)


These equations are quadratic and that allows us to use the operator inference package. Read more about the lifting process here: {mod}`opinf.lift`.

The following code cell defines our variable transformation.

In [ ]:
class EulerLifter(opinf.lift.LifterTemplate):
    """Variable transformations for the Euler equations between the
    conservative variables and the specific volume variables.
    """

    num_original_variables = 3
    num_lifted_variables = 3

    def lift(self, original_state):
        """Map the conservative variables to the specific volume variables,
        [rho, rho*v, rho*e] -> [v, p, 1/rho].
        """
        rho, rho_v, rho_e = np.split(
            original_state,
            self.num_original_variables,
            axis=0,
        )

        v = rho_v / rho
        p = (gamma - 1) * (rho_e - 0.5 * rho_v * v)  # From the ideal gas law.
        zeta = 1 / rho

        return np.concatenate((v, p, zeta))

    def unlift(self, lifted_state):
        """Map the specific volume variables to the conservative variables,
        [v, p, 1/rho] -> [rho, rho*v, rho*e].
        """
        v, p, zeta = np.split(
            lifted_state,
            self.num_lifted_variables,
            axis=0,
        )

        rho = 1 / zeta
        rho_v = rho * v
        rho_e = p / (gamma - 1) + 0.5 * rho_v * v  # From the ideal gas law.

        return np.concatenate((rho, rho_v, rho_e))

We now apply the variable transformation to the training snapshots.
Recall that `Q_fom` contains several training trajectories, each of which corresponds to different initial conditions.
It will be helpful for later on to keep track of the trajectories separately instead of concatenating them into a single snapshot matrix.

In [ ]:
euler_lifter = EulerLifter()
Q_fom_lifted = np.array([euler_lifter.lift(Q) for Q in Q_fom])
Q_fom_lifted.shape

**Important**: In this example, we start with three state variables and apply a variable transformation to get three different state variables. However, this step is often called "lifting" because it is possible to introduce more variables than we started with through the transformation.

### Center and/or Scale

The variables $\vec{q} = (v, p, \zeta)$ have significantly different characteristic scales, which makes it difficult to equally represent all three variables with a single low-dimensional approximation.

In [ ]:
def data_ranges(snapshots):
    for variable, name in zip(
        np.split(np.hstack(snapshots), 3),
        ("velocity", "pressure", "spec.vol"),
    ):
        print(f"Range({name}) = [{variable.min():.2e}, {variable.max():.2e}]")


data_ranges(Q_fom_lifted)

To address this issue, we non-dimensionalize (rescale) the training data so that each variable has the same characteristic scale.
In this example, we select characteristic scales $\bar{v} = 100~[\text{m}/\text{s}]$ for the velocity and $\bar{p} = 10^5~[\text{Pa}=\text{kg}/\text{ms}^2]$ for the pressure.
A corresponding characteristic scale for specific volume can then be determined from the ideal gas law {eq}`eq_ideal_gas_law`:

$$
\begin{aligned}
    \bar{\zeta}
    = \frac{\bar{v}^2}{\bar{p}}
    = \frac{(100)^2~[\text{m}^{2}/\text{s}^{2}]}{10^5~[\text{kg}/\text{ms}^2]}
    = \frac{1}{10}~[\text{m}^3/\text{kg}].
\end{aligned}
$$

New, scaled variables are then defined as

$$
\begin{aligned}
    v' = v / \bar{v},
    \qquad
    p' = p / \bar{p},
    \qquad
    \zeta' = \zeta / \bar{\zeta}.
\end{aligned}
$$

The following class implements this non-dimensionalization.

In [ ]:
# Characteristic scales for each variable.
v_bar = 1e2
p_bar = 1e5
z_bar = 1e-1  # = v_bar**2 / p_bar

euler_scaler = opinf.pre.TransformerMulti(
    transformers=[
        opinf.pre.ScaleTransformer(1 / v_bar, name="velocity"),
        opinf.pre.ScaleTransformer(1 / p_bar, name="pressure"),
        opinf.pre.ScaleTransformer(1 / z_bar, name="specific volume"),
    ]
).fit(Q_fom_lifted[0])

In [ ]:
# Rescale the variables in each training trajectory.
Q_fom_scaled = np.array([euler_scaler.transform(Q) for Q in Q_fom_lifted])

data_ranges(Q_fom_scaled)

:::{admonition} Important
:class: important
There are many ways to preprocess data appropriately for the purpose of learning reduced models.

- A common preprocessing step is to center the data around zero before applying a scaling. Note that this can change the form of the ROM to be learned. An example of this is shown later on in this notebook.
- The scaling applied above is based on a brief dimensional analysis of the lifted variables. In more complex settings, it may be more convenient to apply a data-driven scaling (e.g., dividing each variable by its mean value) or to target a certain range for the scaled variables, usually $[-1, 1]$ or $[0, 1]$. In the {mod}`opinf` package, this can be accomplished using {class}`opinf.pre.ShiftScaleTransformer` instead of {class}`opinf.pre.ScaleTransformer`.

:::

## Reduce Data Dimensionality

We now have discretized, lifted, scaled snapshot data stored in the array `Q_fom_scaled`.
Our goal now is to approximate the preprocessed state $\mathbf{q}(t)\in\mathbb{R}^{n},$ $n = 3n_x,$ with only $r$ degrees of freedom for some small $r \ll n$.
That is, we want a matrix $\mathbf{V}_{\!r}\in\mathbb{R}^{n \times r}$ with orthonormal columns such that $\mathbf{q}(t) \approx \mathbf{V}_{\!r}\hat{\mathbf{q}}(t)$ for some $\hat{\mathbf{q}}(t)\in\mathbb{R}^{r}.$

### Compute a POD Basis

A common choice for $\mathbf{V}_{\!r}$ is the proper orthogonal decomposition (POD) basis, obtained from the singular value decomposition (SVD) of the training data.
Let $\mathbf{Q}\in\mathbb{R}^{n \times k}$ be the matrix of all preprocessed snapshots as columns.
The SVD of $\mathbf{Q}$ is a matrix factorization

$$
\begin{aligned}
    \mathbf{Q} = \boldsymbol{\Phi\Sigma\Psi}^\mathsf{T},
\end{aligned}
$$

where $\boldsymbol{\Phi}\in\mathbb{R}^{n \times n}$ has orthonormal columns, $\boldsymbol{\Sigma}\in\mathbb{R}^{n \times n}$ is diagonal, and $\boldsymbol{\Psi}^\mathsf{T}\in\mathbb{R}^{n \times k}$ has orthonormal rows.
Then $\mathbf{V}_{\!r}$ is defined to be the first $r$ columns of $\boldsymbol{\Phi}$, i.e., the first $r$ principal left singular vectors of the matrix of preprocessed training snapshots.
This computation is taken care of under the hood with {class}`opinf.basis.PODBasis`.

In [ ]:
Q = np.hstack(Q_fom_scaled)
euler_basis = opinf.basis.PODBasis(num_vectors=50).fit(Q)

:::{admonition} Higher Dimensional Problems 
:class: note

In larger problems, it is common for the dimension of the preprocessed training snapshots to be much larger than the number of training snapshots, i.e., $n \gg k$. In this case, $\boldsymbol{\Phi}\in\mathbb{R}^{n \times k}$, $\boldsymbol{\Sigma}\in\mathbb{R}^{k \times k}$, and $\boldsymbol{\Psi}^\mathsf{T}\in\mathbb{R}^{k \times k}.$ An alternative to the SVD in this scenario is to eigendecompose $\mathbf{Q}^\mathsf{T}\mathbf{Q}\in\mathbb{R}^{k \times k}$ to get the first $k$ singular vectors of $\mathbf{Q}.$ This strategy is known as the _method of snapshots_ in the literature.

:::

### Select the Reduced Dimension

One way to select the reduced dimension $r$ when using POD is to examine the singular values of the training snapshots, the diagonal values of $\boldsymbol{\Sigma}$.

In [ ]:
euler_basis.plot_svdval_decay(right=20.5, linestyle=":")
plt.xticks(range(2, 21, 2))
plt.show()

In this example, we choose $r = 9$ because of the large singular value gap between $r = 9$ and $r = 10$.

<!-- In classical model reduction (e.g., balanced truncation), we never truncate the modes in the middle of a plateau of singular values because the modes in a plateau all have similar physical relevance. -->

In [ ]:
def plot_basis_vectors(x, basis_matrix, sharey: bool = True):
    """Plot basis vectors over the spatial domain for each state variable."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=sharey)
    for basis_chunk, ax in zip(np.split(basis_matrix, 3, axis=0), axes):
        for i in range(basis_matrix.shape[1]):
            ax.plot(x, basis_chunk[:, i], label=rf"basis vector ${i+1}$")
        ax.set_xlim(x[0], x[-1])

    axes[0].set_ylabel("velocity basis vectors")
    axes[1].set_ylabel("pressure basis vectors")
    axes[2].set_ylabel("spec.vol basis vectors")
    axes[2].set_xlabel(r"$x\in[0,L]$")


    fig.subplots_adjust(hspace=0.2, right=0.8)
    axes[0].legend(
        loc="center right",
        bbox_to_anchor=(1, 0.5),
        bbox_transform=fig.transFigure,
    )
    fig.align_ylabels(axes)
    plt.show()

In [ ]:
euler_basis.set_dimension(num_vectors=9)

# Plot the columns of the basis matrix.
plot_basis_vectors(x, euler_basis.entries)
plt.show()

Solutions of any ROM that uses the basis matrix $\mathbf{V}_{\!r}$ are restricted to the span of these basis vectors.

### Unexample: What if the Data are NOT Scaled?

**Computing** a basis from *unscaled* data can result in a poor low-dimensional state approximation and eventual issues in the ROM.
The next block computes a POD basis from the lifted but unscaled data.

In [ ]:
# Compute a POD basis from the lifted (but unscaled) data.
unscaled_basis = opinf.basis.PODBasis(
    num_vectors=euler_basis.reduced_state_dimension
).fit(np.hstack(Q_fom_lifted))

# Visualize the resulting basis.
plot_basis_vectors(x, unscaled_basis.entries, sharey=False)

As expected there is a drastic difference in the scales between the velocity, specific volume, and pressure basis vectors with the scales for these vectors being  $[−0.0005, 0.0005],[-2.5 \times 10^7, 7.5 \times 10^7]$,  and  $[−0.1, 1]$,  respectively. Because the characteristic scale of the pressure variable is so much larger than the characteristic scale for velocity and specific volume, the basis vectors for velocity and specific volume are smothered. The problem is also apparent when examining the reconstruction of each variable.

In [ ]:
# Try to reconstruct the lifted data and calculate the error.
Q_fom_lifted_all = np.hstack(Q_fom_lifted)
relative_error_bad_all = unscaled_basis.projection_error(Q_fom_lifted_all)
print(f"Relative recon. error of all variables: {relative_error_bad_all:.2%}")

# Examine the reconstruction error for the individual state variables.
Q_compressed_bad = unscaled_basis.compress(Q_fom_lifted_all)
for i, (full_variable, Vr_bad_var) in enumerate(
    zip(
        np.split(Q_fom_lifted_all, 3, axis=0),
        np.split(unscaled_basis.entries, 3, axis=0),
    )
):
    var_projected_bad = Vr_bad_var @ Q_compressed_bad
    var_error = la.norm(full_variable - var_projected_bad)
    relative_var_error = var_error / la.norm(full_variable)
    print(
        f"Relative recon. error for variable {i+1}: {relative_var_error:.2%}"
    )

The pressure is represented well with an error of $0.01\%$, but velocity and specific volume are not, having errors of $2.95\%$ and $6.51\%$, respectively. Plotting the reconstruction shows the extent of the issue.

In [ ]:
# Project one training trajectory with the unscaled basis.
Q_lifted0 = Q_fom_lifted[0]
Q_lifted0_badprojection = unscaled_basis.project(Q_lifted0)

fig, axes = plot_traces(x, t_train, Q_lifted0, lifted=True)
axes[0].set_title("Original lifted data")

fig, axes = plot_traces(x, t_train, Q_lifted0_badprojection, lifted=True)
axes[0].set_title("Reconstruction of lifted data")
plt.show()

The above graphs are the best possible approximation in terms of error. So no matter how 'good' we make $\hat{\mathbf{q}}$, we cannot get a smaller error percentage with our ROM.

## Learn the Reduced Order Model

Going back to our scaled and centered data, we need to create a ROM. These steps will be the same as those of the other tutorials. We will provide a rough overview of these steps, but for a more detailed walkthrough see the first [getting started](https://operator-inference.github.io/opinf/source/tutorials/basics.html) tutorial. We will be covering the steps from data compression onward.


### Compress Pre-Processed Data

With a basis $\mathbf{V}_{\!r}\in\mathbb{R}^{n \times r}$ in hand, we compress the training snapshots by finding, for each training snapshot $\mathbf{q}_j\in\mathbb{R}^{n}$, the best latent coordinates $\hat{\mathbf{q}}_j\in\mathbb{R}^{r}$ minimizing the reconstruction error $\|\mathbf{q}_j - \mathbf{V}_{\!r}\hat{\mathbf{q}}_j\|_2$.
Because $\mathbf{V}_{\!r}$ has orthonormal columns, this turns out to be $\hat{\mathbf{q}}_j = \mathbf{V}_{\!r}^\mathsf{T}\mathbf{q}_j$.

In [ ]:
Q_compressed = np.array([euler_basis.compress(Q) for Q in Q_fom_scaled])

print(f"Scaled FOM snapshots array: {Q_fom_scaled.shape=}")
print(f"Compressed snapshots array: {Q_compressed.shape=}")
print(
    f"Dimension reduction: n={euler_basis.full_state_dimension}"
    f" -> r={euler_basis.reduced_state_dimension}"
)

### Sanity Check: Reconstruction Error

The reconstruction error of the training snapshots, defined by

$$
\begin{aligned}
    \sum_{j}\|\mathbf{q}_j - \mathbf{V}_{\!r}\hat{\mathbf{q}}_j\|_2
    = \sum_{j}\|\mathbf{q}_j - \mathbf{V}_{\!r}\mathbf{V}_{\!r}^\mathsf{T}\mathbf{q}_j\|_2,
\end{aligned}
$$

gives a sense for how faithfully this basis can approximate the true state.

In [ ]:
relative_recon_error = euler_basis.projection_error(Q, relative=True)
print(f"Relative reconstruction error: {relative_recon_error:.2%}")

### Learn evolution equations

We now have compressed state data, instances of the reduced state variable $\hat{\mathbf{q}}(t)\in\mathbb{R}^{r}$.
The next step is to learn evolution equations for $\hat{\mathbf{q}}(t)$, i.e., to choose an appropriate $\hat{\mathbf{f}}(t, \hat{\mathbf{q}}(t))$ such that $\frac{\text{d}}{\text{d}t}\hat{\mathbf{q}}(t) \approx \hat{\mathbf{f}}(t, \hat{\mathbf{q}}(t))$.

A FOM based on finite differences for the discretized, lifted, scaled state $\mathbf{q}(t)$ would be quadratic due to the quadratic structure of the PDE {eq}`eq_lifted_pdes` governing the lifted variables,

$$
\begin{aligned}
    \frac{\text{d}}{\text{d}t}\mathbf{q}(t)
    = \mathbf{H}[\mathbf{q}(t) \otimes \mathbf{q}(t)]
\end{aligned}
$$

for some $\mathbf{H}\in\mathbb{R}^{n \times n^2}$, where $\otimes$ denotes the [Kronecker product](https://en.wikipedia.org/wiki/Kronecker_product).
Inserting the approximation $\mathbf{q}(t) \approx \mathbf{V}_{\!r}\hat{\mathbf{q}}(t)$ and using the orthogonality property $\mathbf{V}_{\!r}^\mathsf{T}\mathbf{V}_{\!r} = \mathbf{I}$ yields

$$
\begin{aligned}
    \frac{\text{d}}{\text{d}t}\hat{\mathbf{q}}(t)
    = \hat{\mathbf{H}}[\hat{\mathbf{q}}(t) \otimes \hat{\mathbf{q}}(t)]
\end{aligned}
$$ (eq_dqhat)

where $\hat{\mathbf{H}} = \mathbf{V}_{\!r}^\mathsf{T}\mathbf{H}[\mathbf{V}_{\!r}\otimes\mathbf{V}_{\!r}] \in \mathbb{R}^{r \times r^2}$.
However, we cannot compute $\hat{\mathbf{H}}$ this way because we do not have $\mathbf{H}$.

The OpInf approach to learning {eq}`eq_dqhat` is to infer a suitable $\hat{\mathbf{H}}$ by solving the following linear least-squares regression of the compressed training data:

$$
\begin{aligned}
    \min_{\hat{\mathbf{H}}\in\mathbb{R}^{r \times r^2}}
    \sum_{j=1}^{K}\left\|
        \hat{\mathbf{H}}[\hat{\mathbf{q}}_j\otimes\hat{\mathbf{q}}_j]
        - \dot{\hat{\mathbf{q}}}_j
    \right\|_2^2
    + \lambda^2\left\|\hat{\mathbf{H}}\right\|_F^2
\end{aligned}
$$ (eq_least_squares)

where the $\hat{\mathbf{q}}_j\in\mathbb{R}^{r}$ are the compressed training snapshots, $\dot{\hat{\mathbf{q}}}_j\in\mathbb{R}^{r}$ are corresponding time derivatives of the snapshots, and $\lambda \ge 0$ is a regularization hyperparameter.

### Estimate Time Derivatives

To set up {eq}`eq_least_squares`, we estimate the time derivatives of the compressed training snapshots using finite differences.
**Having accurate time derivatives is crucial**; poor time derivative estimates will result in a poor ROM.
Because the original FOM was solved with a fourth-order explicit scheme (RK45), we estimate the derivatives with fourth-order forward differences:

$$
\begin{aligned}
    \frac{\text{d}}{\text{d}t}\hat{\mathbf{q}}(t)\bigg|_{t = t_j}
    \approx \frac{1}{12\delta t}(-25\hat{\mathbf{q}}(t_j) + 48\hat{\mathbf{q}}(t_{j} + \delta t) - 36\hat{\mathbf{q}}(t_{j} + 2\delta t)
    + 16\hat{\mathbf{q}}(t_{j} + 3\delta t) - 3\hat{\mathbf{q}}(t_{j} + 4\delta t)).
\end{aligned}
$$

Several snapshots in each trajectory must also be stripped off since the forward finite difference scheme does not provide estimates for the time derivative at the last four time steps.

In [ ]:
euler_ddts = opinf.ddt.UniformFiniteDifferencer(
    time_domain=t_train,
    scheme="fwd4",
)

Qhats, Qhatdots = zip(*[euler_ddts.estimate(Q) for Q in Q_compressed])
Qhat = np.hstack(Qhats)
Qhatdot = np.hstack(Qhatdots)

print(f"Compressed snapshots array:     {Qhat.shape=}")
print(f"Compressed ddts array:       {Qhatdot.shape=}")

### Solve the Least-squares Regression

We now use the following `opinf` objects to represent the following mathematical structures.

| object                             | what it represents |
| :--------------------------------- | :----------------- |
| {class}`opinf.models.ContinuousModel`     | the system {eq}`eq_dqhat` |
| {class}`opinf.operators.QuadraticOperator` | the term $\hat{\mathbf{q}} \mapsto \hat{\mathbf{H}}[\hat{\mathbf{q}}\otimes\hat{\mathbf{q}}]$ in {eq}`eq_dqhat` |
| {class}`opinf.lstsq.L2Solver`             | the least-squares problem {eq}`eq_least_squares`, including $\lambda$ |

The next block uses these objects to set up and solve {eq}`eq_least_squares`, which defines the system {eq}`eq_dqhat`.

In [ ]:
euler_model = opinf.models.ContinuousModel(
    operators=[opinf.operators.QuadraticOperator()],
    solver=opinf.lstsq.L2Solver(regularizer=1e-8),
)

euler_model.fit(states=Qhat, ddts=Qhatdot)
print(euler_model)

### Solve the ROM

The system {eq}`eq_dqhat` can now be solved by calling [`model.predict()`](opinf.models.ContinuousModel.predict) with an initial (reduced) state vector and a time domain.

### Sanity Check: Reproduce Training Data

Before using the ROM for prediction, we use the training initial conditions and compare the ROM solutions to the compressed training data.
This is an important sanity check: if the ROM cannot reproduce the training data, it is unlikely to be able to issue accurate predictions beyond training scenarios.
Here, an $L^2$ space-time norm is used via {func}`opinf.post.Lp_error()`.

In [ ]:
print("Relative training errors in the reduced space")

_start = time.time()
for i, Qhat in enumerate(Q_compressed):
    qhat0 = Qhat[:, 0]
    Qhat_rom = euler_model.predict(qhat0, t_train)
    rom_error = opinf.post.Lp_error(Qhat, Qhat_rom, t_train)[1]
    print(f"  Trajectory {i: >2d}: {rom_error:.4%}")
rom_solve_time = time.time() - _start

print(f"{len(Q_compressed)} ROM solves in {rom_solve_time:.6} s")

In [ ]:
def plot_reduced_trajectories(t, trajectories):
    """Plot reduced state coordinates in time for two trajectories."""
    Qhat, Qhat_rom = trajectories

    fig, axes = plt.subplots(3, 3, sharex=True, figsize=(12, 8))

    handles, labels = [], []

    for i, ax in enumerate(axes.flat):
        ax.plot(t, Qhat[i, :], "-", label="training data")
        ax.plot(t, Qhat_rom[i, :], "--", label="ROM solution")
        ax.set_ylabel(rf"$\hat{{q}}_{{{i+1}}}(t)$")

        dif_color = "C2"
        difference = np.abs(Qhat[i, :] - Qhat_rom[i, :])
        ax_dif = ax.twinx()
        ax_dif.plot(t, difference, alpha=0.5, color=dif_color, label="pointwise difference")
        max_dif = np.max(difference)
        ax_dif.set_ylim(0, 8 * max_dif)
        ax_dif.set_ylabel(rf"$|q_{{{i+1}}}(t)-\hat{{q}}_{{{i+1}}}(t)|$")

        if i == 0:
            h1, l1 = ax.get_legend_handles_labels()
            h2, l2 = ax_dif.get_legend_handles_labels()
            handles = h1 + h2
            labels = l1 + l2

    for ax in axes[-1, :]:
        ax.set_xlabel(r"time $t$")

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.15)

    fig.legend(
        handles, labels,
        ncol=3,
        loc="lower center",
        bbox_to_anchor=(0.5, 0)
    )

    return fig, axes

In [ ]:
plot_reduced_trajectories(t_train, [Qhat, Qhat_rom])
plt.show()

Looking at the graphs, we can see that our ROM is a good approximation for the training data. We can now move on to predictions for new initial conditions.

### Issue Predictions for New Conditions

Now we take a new initial condition which was **not** used for training, solve the ROM, and compare it with the FOM solution.
We need to remember to do (then undo) all preprocessing steps:

- The new initial condition must be lifted, scaled, and compressed before it is fed to the ROM.
- ROM solutions must be decompressed, unscaled, and unlifted before comparing with the FOM.

The [`opinf.ROM`](opinf.roms.ROM) class joins these pre- and post-processing steps with the learned model.

In [ ]:
euler_rom = opinf.ROM(
    lifter=euler_lifter,
    transformer=euler_scaler,
    basis=euler_basis,
    ddt_estimator=euler_ddts,
    model=euler_model,
)

Finally, we test the accuracy of the ROM on this new initial condition.

In [ ]:
def plot_regimes(
    x,
    t_train,
    t_all,
    states,
    nlocs: int = 20,
    lifted: bool = False,
    title: str = "",
):
    """Plot traces in time and label training/prediction regimes."""
    fig, axes = plot_traces(x, t_all, states, nlocs, lifted)

    for ax in axes.flat:
        ax.axvline(t_train[-1], color="black", linewidth=1)
    ymax = axes[0].get_ylim()[1]
    axes[0].text(t_train[-1] / 2, ymax, "training regime", ha="center")
    xpred = (t_all[-1] - t_train[-1]) / 2 + t_train[-1]
    axes[0].text(xpred, ymax, "prediction regime", ha="center")
    if title:
        axes[0].set_title(title, fontsize="large")

    return fig, axes

In [ ]:
def rom_test_error(rom, plot: bool = False):
    """Solve a ROM at the test initial condition
    and compare it to the FOM solution.
    """
    # Solve the ROM from the test initial condition.
    _start = time.time()
    rom_solution = rom.predict(data["test_init"], t_all)
    rom_solve_time = time.time() - _start
    print(f"ROM solve done in {rom_solve_time:.6f} s")

    # Check the overall ROM error.
    fom_solution = data["test_solution"]
    test_error = opinf.post.Lp_error(fom_solution, rom_solution, t_all)[1]
    print(f"Overall relative ROM test error: {test_error:.2%}")

    # Check the ROM error for each of the physical variables.
    for i, (rom_var, fom_var) in enumerate(
        zip(
            np.split(rom_solution, 3, axis=0),
            np.split(fom_solution, 3, axis=0),
        )
    ):
        test_var_error = opinf.post.Lp_error(fom_var, rom_var, t_all)[1]
        print(f"ROM test error of variable {i+1}: {test_var_error:.2%}")

    if plot:
        plot_regimes(x, t_train, t_all, fom_solution, title="FOM")
        plot_regimes(x, t_train, t_all, rom_solution, title="ROM")
        plt.show()

In [ ]:
rom_test_error(euler_rom, plot=True)

## Supplementary Topics

### Streamline with opinf.ROM

The following block wraps all of the pre-processing and model specification steps together using [`opinf.ROM`](opinf.roms.ROM).

In [ ]:
rom = opinf.ROM(
    lifter=EulerLifter(),
    transformer=opinf.pre.TransformerMulti(
        [
            opinf.pre.ScaleTransformer(1 / v_bar),
            opinf.pre.ScaleTransformer(1 / p_bar),
            opinf.pre.ScaleTransformer(1 / z_bar),
        ]
    ),
    basis=opinf.basis.PODBasis(num_vectors=9),
    ddt_estimator=opinf.ddt.UniformFiniteDifferencer(t_train, scheme="fwd4"),
    model=opinf.models.ContinuousModel(
        operators=[opinf.operators.QuadraticOperator()],
        solver=opinf.lstsq.L2Solver(regularizer=1e-8),
    ),
).fit(Q_fom)

In [ ]:
rom_test_error(rom, plot=False)

### Bonus 1: Regularization Selection

The quality of an operator inference ROM can be sensitive to the choice of the regularization hyperparameter $\lambda$ from {eq}`eq_least_squares`. The following routine selects a $\lambda$ that is optimal in an _a-posteriori_ sense in that it minimizes the training reconstruction error.

In [ ]:
rom.fit_regselect_continuous(
    candidates=np.logspace(-12, 2, 15),
    train_time_domains=t_train,
    states=Q_fom,
    verbose=True,
)

In [ ]:
rom_test_error(rom, plot=False)

In this case, tuning $\lambda$ results in a ROM that is only slightly more accurate, but in larger problems choosing $\lambda$ (or, more generally, the regularization strategy and hyperparameters) can have a significant impact on accuracy and generalizability.

### Bonus 2: Alternative Model Form due to Centering

A common pre-processing step is to center snapshot data before applying scaling or dimension reduction.
In this case, the low-dimensional approximation for the lifted state, formerly $\mathbf{q}(t) \approx \mathbf{V}_{\!r}\hat{\mathbf{q}},$ is now $\mathbf{q}(t) \approx \bar{\mathbf{q}} + \mathbf{V}_{\!r}\hat{\mathbf{q}}$ for some constant $\bar{\mathbf{q}}\in\mathbb{R}^{n}$.
In this case, approximating the purely quadratic dynamics

$$
\begin{aligned}
    \frac{\text{d}}{\text{d}t}\mathbf{q}(t)
    = \mathbf{H}[\mathbf{q}(t) \otimes \mathbf{q}(t)]
\end{aligned}
$$

results in the system

$$
\begin{aligned}
    \frac{\text{d}}{\text{d}t}\hat{\mathbf{q}}(t)
    &= \mathbf{V}_{\!r}^\mathsf{T}\mathbf{H}[(\bar{\mathbf{q}} + \mathbf{V}_{\!r}\hat{\mathbf{q}}) \otimes (\bar{\mathbf{q}} + \mathbf{V}_{\!r}\hat{\mathbf{q}})]
    \\
    &= \mathbf{V}_{\!r}^\mathsf{T}\mathbf{H}[\bar{\mathbf{q}} \otimes \bar{\mathbf{q}}]
    + \mathbf{V}_{\!r}^\mathsf{T}\mathbf{H}[\bar{\mathbf{q}} \otimes \mathbf{V}_{\!r}\hat{\mathbf{q}}]
    + \mathbf{V}_{\!r}^\mathsf{T}\mathbf{H}[\mathbf{V}_{\!r}\hat{\mathbf{q}} \otimes \bar{\mathbf{q}}]
    + \mathbf{V}_{\!r}^\mathsf{T}\mathbf{H}[\mathbf{V}_{\!r}\hat{\mathbf{q}} \otimes \mathbf{V}_{\!r}\hat{\mathbf{q}}],
\end{aligned}
$$

which can be written as a quadratic system with additional lower-order terms,

$$
\begin{aligned}
    \frac{\text{d}}{\text{d}t}\hat{\mathbf{q}}(t)
    = \hat{\mathbf{c}}
    + \hat{\mathbf{A}}\hat{\mathbf{q}}(t)
    + \hat{\mathbf{H}}[\hat{\mathbf{q}} \otimes \hat{\mathbf{q}}].
\end{aligned}
$$

Because of this, if the data are centered, the OpInf regression should be modified to include the constant and linear terms $\hat{\mathbf{c}}$ and $\hat{\mathbf{A}}$:

$$
\begin{aligned}
    \min_{\hat{\mathbf{c}}\in\mathbb{R}^{r},\hat{\mathbf{A}}\in\mathbb{R}^{r\times r},\hat{\mathbf{H}}\in\mathbb{R}^{r \times r^2}}
    \sum_{j=1}^{K}\left\|
        \hat{\mathbf{c}}
        + \hat{\mathbf{A}}\hat{\mathbf{q}}_j
        + \hat{\mathbf{H}}[\hat{\mathbf{q}}_j\otimes\hat{\mathbf{q}}_j]
        - \dot{\hat{\mathbf{q}}}_j
    \right\|_2^2
    + \lambda^2\left\|\hat{\mathbf{H}}\right\|_F^2.
\end{aligned}
$$

The following code centers the data before scaling.
Instead of non-dimensionalizing the data exactly, each variable is scaled so that the entries of the training states lie in the interval $[0, 1]$.

In [ ]:
# Define the ROM, together with all pre-processing pieces.
rom_centered = opinf.ROM(
    lifter=EulerLifter(),
    transformer=opinf.pre.TransformerMulti(  # Variable-specific scaling
        [
            opinf.pre.ShiftScaleTransformer(
                centering=True,
                scaling="minmax",
                name="velocity",
            ),
            opinf.pre.ShiftScaleTransformer(
                centering=True,
                scaling="minmax",
                name="pressure",
            ),
            opinf.pre.ShiftScaleTransformer(
                centering=True,
                scaling="minmax",
                name="specific volume",
            ),
        ]
    ),
    basis=opinf.basis.PODBasis(num_vectors=9),
    ddt_estimator=opinf.ddt.UniformFiniteDifferencer(t_train, scheme="fwd4"),
    model=opinf.models.ContinuousModel(
        operators=[
            opinf.operators.ConstantOperator(),  # c
            opinf.operators.LinearOperator(),  # Aq(t)
            opinf.operators.QuadraticOperator(),  # H[q(t) ⊗ q(t)]
        ],
        solver=opinf.lstsq.L2Solver(regularizer=1e-8),
    ),
)

# Fit the ROM to the training data, selecting the regularization automatically.
rom_centered.fit_regselect_continuous(
    candidates=np.logspace(-12, 2, 15),
    train_time_domains=t_train,
    states=Q_fom,
)

# Evaluate the ROM on the testing data.
rom_test_error(rom_centered, plot=False)

On a related note, the principal POD basis vector can often be interpreted as having a centering effect.
Note that the gap between the first two singular values is smaller than it was for the unscaled data.
The singular value decay shows that $r = 8$ is a good choice now.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
rom.basis.plot_svdval_decay(right=20.5, linestyle=":", ax=axes[0])
rom_centered.basis.plot_svdval_decay(right=20.5, linestyle=":", ax=axes[1])

for ax in axes:
    ax.set_xticks(range(2, 21, 2))
    ax.set_ylim(1e-4, 2e0)
axes[0].set_title("Without centering")
axes[1].set_title("With centering")
plt.show()

The gap is eliminated entirely when the data are nondimensionalized exactly, in which case $r = 8$ is the suitable choice.

In [ ]:
rom_centered2 = opinf.ROM(
    lifter=EulerLifter(),
    transformer=opinf.pre.TransformerPipeline(
        [
            opinf.pre.ShiftScaleTransformer(centering=True),  # Center.
            euler_scaler,  # Non-dimensionalize exactly.
        ],
    ),
    basis=opinf.basis.PODBasis(num_vectors=8),
    ddt_estimator=opinf.ddt.UniformFiniteDifferencer(t_train, scheme="fwd4"),
    model=opinf.models.ContinuousModel(
        operators=[
            opinf.operators.ConstantOperator(),
            opinf.operators.LinearOperator(),
            opinf.operators.QuadraticOperator(),
        ],
        solver=opinf.lstsq.L2Solver(regularizer=1e-8),
    ),
)

# Fit the ROM to the training data, selecting the regularization automatically.
rom_centered2.fit_regselect_continuous(
    candidates=np.logspace(-12, 2, 15),
    train_time_domains=t_train,
    states=Q_fom,
)

# Evaluate the ROM on the testing data.
rom_test_error(rom_centered2, plot=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
rom.basis.plot_svdval_decay(right=20.5, linestyle=":", ax=axes[0])
rom_centered2.basis.plot_svdval_decay(right=20.5, linestyle=":", ax=axes[1])

for ax in axes:
    ax.set_xticks(range(2, 21, 2))
    ax.set_ylim(1e-4, 2e0)
axes[0].set_title("Without centering")
axes[1].set_title("With centering")
plt.show()

### Bonus 3: Unexample (Data NOT Scaled)

The [`opinf.ROM`](opinf.roms.ROM) class makes it easy to see what might happen if we try to learn a ROM without scaling the data first as discussed earlier.

In [ ]:
rom_noscaling = opinf.ROM(
    lifter=EulerLifter(),
    # no scaling!
    basis=opinf.basis.PODBasis(num_vectors=9),
    ddt_estimator=opinf.ddt.UniformFiniteDifferencer(t_train, scheme="fwd4"),
    model=opinf.models.ContinuousModel(
        operators=[opinf.operators.QuadraticOperator()],
        solver=opinf.lstsq.L2Solver(regularizer=1e-8),
    ),
)

# Try to select an optimal regularization
# (this will fail and raise a RuntimeError).
rom_noscaling.fit_regselect_continuous(
    candidates=np.logspace(-12, 2, 15),
    train_time_domains=t_train,
    states=Q_fom,
    verbose=True,
)

Note that the singular value decay pattern has changed.

In [ ]:
rom_noscaling.basis.plot_svdval_decay(right=20, linestyle=":")
plt.show()

In this case, we are able to set up and solve the inference problem {eq}`eq_least_squares` and thereby define a reduced model {eq}`eq_dqhat`, but that model is not stable.
Decreasing the number of basis vectors to $r = 3$ results in a stable model, but the accuracy is poor because the underlying basis is not expressive enough to faithfully represent true solutions.

In [ ]:
rom_noscaling.basis.set_dimension(num_vectors=3)

rom_noscaling.fit_regselect_continuous(
    candidates=np.logspace(-12, 2, 15),
    train_time_domains=t_train,
    states=Q_fom,
    verbose=True,
)

In [ ]:
rom_test_error(rom_noscaling, plot=True)